# Fast Baseline - Clasificación de décadas

Clasificador de textos históricos por década usando fine-tuning de DistilBERT.

## 1. Importaciones

In [1]:
# Librerias
import re
import nltk
import torch

import pandas as pd
import numpy as np
import lightgbm as lgb

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from nltk.corpus import stopwords

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import torch.nn as nn

from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

c:\Users\Juan David\Downloads\MaterialDeClase-ISIS-2611\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carga y exploración de datos

In [2]:
# Cargar datos
train = pd.read_csv('./../data/train.csv')
eval_df = pd.read_csv('./../data/eval.csv')

data = train.copy()

print("TAMAÑO DEL DATASET")
print(data.shape)
print(data.head())

print("\nDISTRIBUCIÓN DE LAS DECADAS")
print(data['decade'].value_counts())
print("\nNúmero de decadas:", data['decade'].nunique())

TAMAÑO DEL DATASET
(31403, 2)
                                                text  decade
0  \nHonorarias ¡jubiladas. 57 \ndit.ad Pontem de...     164
1  gone. Sus amigos , sus clientes, todo \ncuanto...     182
2  Prefosen quemanera,e per qualesfolpechas deuan...     157
3  Caistro  el  M  a  y  o  r  a  i  .]  Del  ape...     163
4  \nlos  que  panden  macho  ;  y \notros  en  l...     166

DISTRIBUCIÓN DE LAS DECADAS
decade
160    848
172    842
155    836
170    833
167    831
178    831
154    830
157    827
163    827
180    825
168    822
175    817
171    816
165    814
151    812
188    809
179    809
182    808
162    808
174    807
164    804
185    803
184    802
173    802
159    802
181    795
183    794
156    792
161    787
187    787
150    786
152    785
177    782
166    779
158    778
153    775
186    773
169    771
176    754
Name: count, dtype: int64

Número de decadas: 39


### 2.1 Longitud de los textos

Los textos varían significativamente en tamaño. Es importante conocer esta distribución para definir el `max_length` del tokenizador.

In [3]:
# Varianza en el tamaño de los textos.
data['text_len'] = data['text'].apply(len)
print(data['text_len'].describe())

count    31403.000000
mean       520.568290
std        530.947792
min        120.000000
25%        182.000000
50%        315.000000
75%        643.000000
max       7418.000000
Name: text_len, dtype: float64


### 2.2 Ejemplos por década

Inspección visual de textos representativos de distintas épocas para apreciar las diferencias ortográficas y de estilo.

In [4]:
# Ejemplos de distintas décadas
for decade in [150, 165, 180]:
    ejemplo = data[data['decade'] == decade]['text'].iloc[0]
    print(f"\n=== Década {decade} ===")
    print(ejemplo[:300])
    print("---")


=== Década 150 ===
efiotnl fiiT’e^ pt\»tf)e 4 trCe et lleene.^^ta^ 
41 tT»íi A*e(leee A(ittc(«t>iieii|l>le iié <oii|ette 
*iW íléiW^* temer pw tnner>>ellmpí«íK> 
---

=== Década 165 ===
Efto fé ha viña en el Confejo de Portagal,c\ qual 
pbí M0'fé ; r precedido del de Aragón , nunca ha queri- 
do concurrir éfl 1 asProce fsione s , be lámanos , ni jun- 
---

=== Década 180 ===

(87) 
obligación  que  las  Ordenanzas  imponen  al  Director 
general  de  la  Armada  sobre  el  zelar  que  se  mejoren 
las  cartas  y  derroteros  en  conformidad  de  las  noticias 
que  deben  dársele  en  quanto  á  los  descubrimientos  de 
nuevas  tierras,  islas,  baxos  y  sondas,  d  r
---


## 3. Preprocesamiento de texto

Se normalizan saltos de línea, se eliminan caracteres ruidosos (OCR), se convierten a minúsculas, se remueven stopwords en español y tokens muy cortos.

In [5]:
# Stopwords en español - nltk
nltk.download('stopwords')

stop_words = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    # 1. Normalizar saltos de línea y espacios múltiples
    texto = re.sub(r'\n+', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    
    # 2. Quitar caracteres que claramente son ruido OCR
    # (símbolos que no son letras, números ni puntuación básica)
    texto = re.sub(r'[^\w\s.,;:!?áéíóúüñÁÉÍÓÚÜÑ]', ' ', texto)
    
    # 3. Strip
    texto = texto.strip().lower()

    # 4. Quitar stopwords
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stop_words]

    # 5. Quitar puntuación y números que quedaron sueltos
    texto = re.sub(r'\b[\d.,;:!?]+\b', ' ', texto)
    
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stop_words]
    
    # 6. Filtrar tokens de 1 o 2 caracteres (ruido OCR)
    palabras = [p for p in palabras if len(p) > 2]

    return ' '.join(palabras)

data['text_clean'] = data['text'].apply(limpiar_texto)
eval_df['text_clean'] = eval_df['text'].apply(limpiar_texto)

# Verifica el resultado
for decade in [150, 165, 180]:
    ejemplo = data[data['decade'] == decade]['text_clean'].iloc[0]
    print(f"\n=== Década {decade} ===")
    print(ejemplo[:300])

[nltk_data] Downloading package stopwords to C:\Users\Juan
[nltk_data]     David\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



=== Década 150 ===
efiotnl fiit trce lleene. leee ittc iieii iié oii ette íléiw temer tnner ellmpí

=== Década 165 ===
efto viña confejo portagal qual pbí precedido aragón nunca queri concurrir éfl asproce fsione lámanos jun

=== Década 180 ===
obligación ordenanzas imponen director general armada zelar mejoren cartas derroteros conformidad noticias deben dársele quanto descubrimientos nuevas tierras, islas, baxos sondas, rectificación acaso hiciese posiciones locadas. cuidado duda alguna bueno pro vechoso: capaz producir armada real utili


## 4. Preparación de datos para entrenamiento

### 4.1 Codificación de etiquetas

LabelEncoder convierte las décadas (enteros como 150, 157...) a índices 0..N-1 para usarlos con CrossEntropyLoss.

In [6]:
# ── 2. Encodear las etiquetas (decade -> índice numérico) ────────────────────
# La década viene como int (ej: 157, 162, 197...)
# LabelEncoder la convierte a índices 0, 1, 2, ... N-1
le = LabelEncoder()
data['label'] = le.fit_transform(data['decade'])

print("Clases únicas:", le.classes_)       # décadas originales
print("Número de clases:", len(le.classes_))

Clases únicas: [150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167
 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185
 186 187 188]
Número de clases: 39


### 4.2 División train/validation

Split estratificado al 85/15 para mantener la proporción de clases.

In [7]:
# ── 3. Split train/validation ────────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    data['text_clean'].values,
    data['label'].values,
    test_size=0.15,
    random_state=42,
    stratify=data['label']   # mantener proporción de clases
)

print(f"Train: {len(X_train)} | Val: {len(X_val)}")

Train: 26692 | Val: 4711


### 4.3 Tokenización y clase Dataset

Se usa DistilBERT con `max_length=256`. La clase `DecadeDataset` se encarga de tokenizar los textos al vuelo durante el entrenamiento.

In [8]:
# ── 4. Clase Dataset ─────────────────────────────────────────────────────────
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256   # 256 cubre la mayoría de párrafos sin llegar al límite de 512

class DecadeDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=256):
        self.texts = texts
        self.labels = labels          # None para el set de evaluación
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',     # padear hasta max_len
            truncation=True,          # truncar si excede max_len
            return_tensors='pt'       # retornar tensores de PyTorch
        )

        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

### 4.4 DataLoaders

Se instancian los datasets (train, validation, evaluación) y se envuelven en DataLoaders con batch size de 32.

In [9]:
# ── 5. Instanciar Datasets ───────────────────────────────────────────────────
train_dataset = DecadeDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = DecadeDataset(X_val,   y_val,   tokenizer, MAX_LEN)
eval_dataset  = DecadeDataset(
    eval_df['text_clean'].values,
    labels=None,
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# ── 6. DataLoaders ───────────────────────────────────────────────────────────
BATCH_SIZE = 32   # bajar a 16 si hay OOM en GPU

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
eval_loader  = DataLoader(eval_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# Verificar un batch
batch = next(iter(train_loader))
print("input_ids shape:", batch['input_ids'].shape)       # (32, 256)
print("attention_mask shape:", batch['attention_mask'].shape)
print("labels shape:", batch['labels'].shape)

input_ids shape: torch.Size([32, 256])
attention_mask shape: torch.Size([32, 256])
labels shape: torch.Size([32])


## 5. Definición del modelo

Se usa DistilBERT como backbone congelado (transfer learning) y se agrega una cabeza clasificadora con dropout. Se extrae el token `[CLS]` para la clasificación.

In [10]:
class DecadeClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.3):
        super(DecadeClassifier, self).__init__()

        # Backbone preentrenado (transfer learning aquí)
        self.backbone = AutoModel.from_pretrained(model_name)

        hidden_size = self.backbone.config.hidden_size  # 768 para DistilBERT

        # Cabeza de clasificación encima del [CLS] token
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # El token [CLS] resume toda la oración — es lo que usamos
        cls_output = outputs.last_hidden_state[:, 0, :]  # shape: (batch, 768)
        logits = self.classifier(cls_output)             # shape: (batch, num_classes)
        return logits

NUM_CLASSES = len(le.classes_)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Usando:", device)

model = DecadeClassifier(MODEL_NAME, NUM_CLASSES, dropout=0.3).to(device)

Usando: cpu


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3889.49it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 6. Entrenamiento

### 6.1 Configuración del optimizador

Learning rate bajo (2e-5) con AdamW y weight decay. Se usa un scheduler lineal con warmup del 10% para mejorar la estabilidad al inicio.

In [11]:
EPOCHS = 5
LR = 2e-5   # learning rate bajo — estándar para fine-tuning de Transformers

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

# Scheduler con warmup — mejora estabilidad al inicio del fine-tuning
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

### 6.2 Funciones de entrenamiento y evaluación

- `train_epoch`: forward pass, backprop, grad clipping, métricas.
- `eval_epoch`: solo forward pass con `torch.no_grad()`.

In [12]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # evitar exploding gradients
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

### 6.3 Bucle de entrenamiento

Se entrena por 5 épocas y se guarda el mejor modelo según accuracy en validación.

In [13]:
# ── Training loop completo ───────────────────────────────────────────────────
best_val_acc = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss,   val_acc   = eval_epoch(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Guardar el mejor modelo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"  Mejor modelo guardado (val_acc={val_acc:.4f})")

KeyboardInterrupt: 

## 7. Predicción y submission

Se carga el mejor modelo guardado y se generan predicciones sobre el conjunto de evaluación. Finalmente se exporta un CSV con el formato de submission.

In [ ]:
# Cargar el mejor modelo
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

all_preds = []

with torch.no_grad():
    for batch in eval_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits = model(input_ids, attention_mask)
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

# Convertir índices de vuelta a décadas originales
predicted_decades = le.inverse_transform(all_preds)

submission = pd.DataFrame({
    'id':     eval_df['id'],
    'answer': predicted_decades
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10))